1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path.home() / 'ai_models'

'''
Select the GPU IDs available to this notebook.
指定這份 Notebook 可使用的 GPU ID。
'''
GPU_IDS = [0]


'''
Select the GPUs for the generation model.
指定生成模型使用的 GPU ID。
'''
MODEL_GPU_IDS = [0]


'''
Select the GPU for text encoders, VAE, and codec models.
指定文字編碼器、VAE 與 Codec 等輔助模型使用的 GPU ID。
'''
AUX_GPU_ID = 0


'''
Derive the primary model GPU for existing loaders; no manual edit needed.
依上述設定取得 Loader 使用的主要生成 GPU，不需要手動修改。
'''
MODEL_GPU_ID = MODEL_GPU_IDS[0]


'''
Count the selected GPUs for the existing initialization flow.
計算指定的 GPU 數量，供原有初始化流程使用。
'''
GPU_COUNT = len(GPU_IDS)

'''
Reject duplicate, out-of-range, or inconsistent GPU assignments.
拒絕重複、超出設定範圍或互相衝突的 GPU 編號。
'''
assert GPU_IDS and len(set(GPU_IDS)) == len(GPU_IDS) and all(type(i) is int and i >= 0 for i in GPU_IDS)
assert MODEL_GPU_IDS and len(set(MODEL_GPU_IDS)) == len(MODEL_GPU_IDS)
assert set(MODEL_GPU_IDS).issubset(GPU_IDS) and AUX_GPU_ID in GPU_IDS
ROOT.mkdir(parents=True, exist_ok=True)

'''
Validate the selected IDs against the ROCm/CUDA-visible PyTorch devices.
比對 ROCm／CUDA 版 PyTorch 可見的 GPU 編號，避免錯誤分配。
'''
def validate_torch_gpus(sharding=False):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('找不到 PyTorch GPU。請先在 Ubuntu 安裝符合 GPU 型號的 ROCm PyTorch。')
    available = torch.cuda.device_count()
    if max(GPU_IDS) >= available:
        raise RuntimeError(f'GPU_IDS={GPU_IDS}，但 PyTorch 只看得到 {available} 張 GPU。')
    if len(MODEL_GPU_IDS) > 1 and not sharding:
        raise RuntimeError('這套 Loader 尚未驗證跨 GPU 主模型切分；MODEL_GPU_IDS 請先只指定一張。')
    backend = 'AMD ROCm' if torch.version.hip else 'NVIDIA CUDA'
    print(f'{backend}：選擇 {GPU_IDS}；生成 {MODEL_GPU_IDS}；輔助 cuda:{AUX_GPU_ID}')
    for gpu_id in GPU_IDS:
        memory = torch.cuda.get_device_properties(gpu_id).total_memory / 1024**3
        print(f'GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)} ({memory:.1f} GiB)')


        '''
        Warn if the selected device has less memory than the 32GB profile expects.
        若選用的 GPU 容量低於 32GB 配置需求，提醒可能需要更積極的 Offload。
        '''
        if memory < 27:
            '''
            Explain the available-VRAM limitation without blocking execution.
            提醒可用 VRAM 較少，但不阻擋使用者繼續執行。
            '''
            print('提醒：此 Notebook 預設供約 32GB VRAM 的 GPU 使用；目前容量可能需要更積極的 Offload。')


PROJECT = ROOT / 'yue2.cpp'
SERVER = PROJECT / 'build/yue-server'

'''
Choose where YuE2 backbone and VAE checkpoint files are stored.
設定 YuE2 主模型與 VAE 權重下載資料夾。
'''
MODEL_DIR = ROOT / 'yue2_models'
REPO = 'Serveurperso/YuE2-GGUF'
PORTS = {i: 8091 + i for i in GPU_IDS}  # 各指定 GPU 使用不同 Port

'''
Select the Vulkan GPU for one YuE2 service; do not duplicate the generation workload.
指定 YuE2 服務使用的 Vulkan GPU；同一時間只處理一筆生成工作。
''''''
Use the highest published precision of the 3B backbone and F32 VAE.
選用官方 3B BF16 主模型與 F32 VAE，以維持原始精度。
'''
YUE_BACKBONE_FILE = 'YuE2-3B-BF16.gguf'
YUE_VAE_FILE = 'YuE2-Vae-F32.gguf'
GPU_ID = MODEL_GPU_ID
VULKAN_GPU_ID = GPU_ID  # Vulkan 編號可能與 ROCm 不同；請以 vulkaninfo 確認
YUE_DEVICE_NAME = f'Vulkan{VULKAN_GPU_ID}'
OUTPUT_DIR = ROOT / 'yue2_outputs'
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535

'''
Keep the server identity fixed so clients cannot connect to a different model by mistake.
固定模型服務識別碼，讓客製化 client 可以檢查是否連到正確模型；一般不應修改。
'''
REMOTE_SERVICE_ID = '06_YuE2_音樂生成_Ubuntu'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整

'''
Limit concurrently parsed uploads to protect 主機 RAM.
限制同時解析大型上傳檔案的數量，避免占滿主機 RAM。
'''
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2
REMOTE_STALL_HOURS = 2

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert GPU_ID in PORTS and len(set(PORTS.values())) == len(PORTS)
assert all(1024 <= port <= 65535 for port in PORTS.values())
assert REMOTE_PORT not in PORTS.values(), 'FIFO Port 與 YuE2 服務 Port 不得相同。'
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝環境與匯入套件


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import shutil
import subprocess
import sys
from pathlib import Path

'''
Use the Vulkan backend instead of NVIDIA-only CUDA build tools.
改用 Vulkan 後端，不再檢查 nvidia-smi 或 NVIDIA nvcc。
'''
assert shutil.which('cmake'), '請先在 Ubuntu 安裝 cmake、C++ compiler 與 Vulkan SDK。'
print(f'YuE2 選用：{YUE_DEVICE_NAME}；請確認其 Vulkan 編號符合實際顯示卡。')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'hf_xet', 'requests'],
    check=True,
)

'''
Clone the official source or safely fast-forward an existing checkout.
複製官方原始碼，或以 fast-forward 安全更新現有倉庫。
'''
if not (PROJECT / '.git').exists():
    assert not PROJECT.exists(), f'既有目錄不是 Git 倉庫：{PROJECT}'
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--recurse-submodules',
         'https://github.com/ServeurpersoCom/yue2.cpp.git', str(PROJECT)],
        check=True,
    )
else:
    source_changes = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip()

    '''
    Preserve custom local changes instead of overwriting them.
    保留使用者修改，不覆寫本機資料。
    '''
    if not source_changes:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', '-q'], check=True)
    else:
        print('YuE2 原始碼已有本機修改，為避免衝突未更新；請在新 Session 重新執行。')

subprocess.run(
    ['git', '-C', str(PROJECT), 'submodule', 'update', '--init', '--recursive'],
    check=True,
)
BUILD_JOBS = 4

'''
Track the source revision used to compile the Vulkan server binary.
記錄用來編譯 Vulkan Server 的原始碼版本。
'''
BUILD_STAMP = PROJECT / 'build/.ubuntu_vulkan_yue2_revision'
CURRENT_REVISION = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if not SERVER.is_file() or not BUILD_STAMP.is_file() or BUILD_STAMP.read_text().strip() != CURRENT_REVISION:
    jobs = BUILD_JOBS
    cpu_slots = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 2)
    print(f'Build jobs: {jobs} | Available CPU slots: {cpu_slots}', flush=True)
    if jobs > cpu_slots:
        print('Notice: Build jobs exceed CPU slots; try BUILD_JOBS=6 if slower or memory-limited.', flush=True)
    command = [
        'cmake', '-S', str(PROJECT), '-B', str(PROJECT / 'build'),
        '-DGGML_VULKAN=ON', '-DGGML_CUDA=OFF',
        '-DCMAKE_BUILD_TYPE=Release',
    ]

    '''
    Prefer Ninja for new builds; leave existing Makefiles unchanged.
    新建置優先使用 Ninja，但不更改既有 Makefile 專案。
    '''
    if shutil.which('ninja') and not (PROJECT / 'build/CMakeCache.txt').exists():
        command.extend(['-G', 'Ninja'])
    if shutil.which('ccache'):
        '''
        Enable the cache for both C++ and CUDA compilation.
        讓 C++ 與 CUDA 編譯都使用快取。
        '''
        command.extend([
            '-DCMAKE_CXX_COMPILER_LAUNCHER=ccache',
            
        ])
    subprocess.run(command, check=True)
    print(f'Building yue-server with {jobs} parallel jobs...', flush=True)
    subprocess.run(
        ['cmake', '--build', str(PROJECT / 'build'), '--target',
         'yue-server', '--parallel', str(jobs)],
        check=True,
    )

    '''
    Record the revision only after a successful build.
    只有編譯成功後才記錄完成的原始碼版本。
    '''
    BUILD_STAMP.write_text(CURRENT_REVISION + '\n', encoding='utf-8')
else:
    print('Reusing compiled yue-server; Vulkan build skipped.', flush=True)

'''
Verify the compiled executable and identify the source revision.
確認執行檔存在，並記錄原始碼版本。
'''
assert SERVER.is_file(), 'yue-server 編譯失敗。'
revision = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True,
).strip()
print(f'YuE2 server ready (commit {revision}).')
print('✓ 第 2 節完成：YuE2 環境與服務程式已準備完成。')


3. 下載模型


In [ ]:
'''
Download the original-precision backbone and VAE only once.
只下載一次 BF16 原始精度主模型與 F32 VAE。
'''
from huggingface_hub import hf_hub_download

MODEL_DIR.mkdir(parents=True, exist_ok=True)
BACKBONE = Path(hf_hub_download(
    repo_id=REPO, filename=YUE_BACKBONE_FILE, local_dir=str(MODEL_DIR),
))
VAE = Path(hf_hub_download(
    repo_id=REPO, filename=YUE_VAE_FILE, local_dir=str(MODEL_DIR),
))
assert BACKBONE.stat().st_size > 6 * 1024**3
assert VAE.stat().st_size > 400 * 1024**2
print('Models:', BACKBONE.name, VAE.name)
print('✓ 第 3 節完成：主模型與 VAE 權重已準備完成。')


4. 啟動單 GPU 常駐服務（指定 Vulkan GPU）


In [ ]:
'''
Start one resident model service on the selected GPU.
只在指定 GPU 啟動一個模型常駐服務。
'''
import requests
import time
SERVER_PROCS = globals().get('SERVER_PROCS', {})
assert GPU_ID in PORTS, "GPU_ID 必須在 PORTS 裡。"

'''
Keep the model on one GPU instead of loading two independent copies.
只在單張 GPU 載入模型，不額外複製第二份。
'''
for gpu_id in (GPU_ID,):
    port = PORTS[gpu_id]
    address = f'http://127.0.0.1:{port}'
    process = SERVER_PROCS.get(gpu_id)
    try:
        healthy = requests.get(address + '/health', timeout=2).ok
        props = requests.get(address + '/props', timeout=2).json() if healthy else {}
    except requests.RequestException:
        healthy, props = False, {}
    if healthy:
        assert Path(props.get('model', '')).name == BACKBONE.name, f'Port {port} 已由其他模型占用。'
        print(f'GPU {gpu_id}: reusing resident server.')
        continue

    '''
    Avoid launching a second server while the first is still loading.
    原服務仍在啟動時，避免再啟動第二份模型。
    '''
    if process is not None and process.poll() is None:
        raise RuntimeError(f'GPU {gpu_id} 原服務尚未就緒，請查看 yue2_gpu{gpu_id}.log。')
    command = [
        str(SERVER), '--model', str(BACKBONE), '--vae', str(VAE),
        '--device', YUE_DEVICE_NAME, '--host', '127.0.0.1',
        '--port', str(port), '--keep-loaded', '--max-seq', '8192',
        '--max-batch', '1', '--vae-core', '256',
    ]
    log_path = ROOT / f'yue2_gpu{gpu_id}.log'
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(
            command, cwd=PROJECT, stdout=log,
            stderr=subprocess.STDOUT, start_new_session=True,
        )
    SERVER_PROCS[gpu_id] = process
    for _ in range(120):
        if process.poll() is not None:
            raise RuntimeError(log_path.read_text(errors='replace')[-3000:])
        try:
            if requests.get(address + '/health', timeout=2).ok:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    else:
        raise TimeoutError(f'GPU {gpu_id}: {log_path.read_text(errors="replace")[-2000:]}')
    print(f'GPU {gpu_id}: server ready (port {port}).')

print('BF16 weights load on first generation and remain resident afterward.')
print('✓ 第 4 節完成：單 GPU 常駐服務已啟動（模型首次生成時載入）。')


5. 生成設定


In [ ]:
'''
Describe the musical style and vocal performance.
設定曲風與演唱方式。
'''
STYLE = '''
Mandarin pop, warm female vocal, acoustic guitar, soft piano, ballad, 90 BPM
'''.strip()
LYRICS = '''
[Verse]
微風吹過熟悉的街道
晚霞落在回家的轉角

[Chorus]
讓心裡的光慢慢閃耀
明天還有新的美好
'''.strip()
DURATION = 30
STEPS = 32

'''
Available choices: full, melody, off.
可選值：full、melody、off。
'''

COT = 'full'  # 'off'（僅歌曲）, 'melody'（旋律規劃）, 'full'（完整規劃）
LM_SEED, SEED = -1, -1

'''
Set a filename that will never overwrite previous results.
設定輸出檔名，產生新版本時不覆蓋舊檔。
'''
OUTPUT_NAME = 'yue2_bf16.mp3'
ABC_PATH = ''
SHOW_SCORE = True
EXPORT_MIDI = True
print('✓ 第 5 節完成：歌詞、COT、ABC 及其他參數已設定（尚未生成）。')


6. 生成音樂（可重複執行）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import json
    import time
    import requests
    from pathlib import Path
    from email import policy
    from email.parser import BytesParser
    assert GPU_ID in PORTS, 'GPU_ID 必須在 PORTS 裡。'
    assert COT in ('full', 'melody', 'off') and DURATION > 0 and STEPS > 0
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    '''
    Optionally load a user-edited ABC score to guide music generation.
    視需要讀入使用者修改的 ABC 樂譜作為生成條件。
    '''
    score_text = Path(ABC_PATH).read_text(encoding='utf-8').strip() if ABC_PATH else ''
    assert not score_text or COT != 'off', '提供 ABC_PATH 時，COT 應設定 full 或 melody。'
    request_data = {
        'style': STYLE, 'lyrics': LYRICS, 'abc': score_text,
        'cot': COT, 'duration': DURATION, 'steps': STEPS,
        'lm_seed': LM_SEED, 'seed': SEED, 'lm_batch_size': 1,
        'synth_batch_size': 1, 'output_format': 'mp3', 'mp3_bitrate': 192,
    }

    '''
    Generate on one designated server without starting a new model process.
    使用指定服務生成歌曲，不重新啟動模型行程。
    '''
    def generate_on_gpu(gpu_id):
        '''
        Access this GPU's local API endpoint.
        取得這張 GPU 對應的本機 API。
        '''
        address = f'http://127.0.0.1:{PORTS[gpu_id]}'
        submitted = requests.post(address + '/synth', json=request_data, timeout=60)
        submitted.raise_for_status()
        job_id = submitted.json()['id']
        print(f'GPU {gpu_id}: job {job_id} started', flush=True)
        deadline = time.monotonic() + 3600
        last_report = time.monotonic()
        while time.monotonic() < deadline:
            status_response = requests.get(address + '/job', params={'id': job_id}, timeout=30)
            status_response.raise_for_status()
            status = status_response.json()
            if status.get('status') == 'done':
                break
            if status.get('status') in ('failed', 'cancelled'):
                raise RuntimeError(f'GPU {gpu_id}: {status}')
            if time.monotonic() - last_report >= 30:
                print(f'GPU {gpu_id}: still generating ({status.get("status")})', flush=True)
                last_report = time.monotonic()
            time.sleep(5)
        else:
            raise TimeoutError(f'GPU {gpu_id}: job {job_id} exceeded 1 hour.')
        response = requests.get(
            address + '/job', params={'id': job_id, 'result': 1}, timeout=180,
        )
        response.raise_for_status()
        mime = (
            b'MIME-Version: 1.0\r\nContent-Type: '
            + response.headers['Content-Type'].encode()
            + b'\r\n\r\n' + response.content
        )
        message = BytesParser(policy=policy.default).parsebytes(mime)
        assert message.is_multipart(), f'GPU {gpu_id}: unexpected result format.'
        parts = list(message.iter_parts())
        audio = next(
            (part.get_payload(decode=True) for part in parts
             if part.get_content_type() in ('audio/mpeg', 'audio/mp3')),
            None,
        )
        assert audio and len(audio) > 4096, f'GPU {gpu_id}: empty MP3 response.'
        replay = next(
            (json.loads(part.get_payload(decode=True)) for part in parts
             if part.get_content_type() == 'application/json'),
            None,
        )
        stem = Path(OUTPUT_NAME).stem or 'yue2'
        index = 1
        while (OUTPUT_DIR / f'{stem}_gpu{gpu_id}_{index:03d}.mp3').exists():
            index += 1
        output = OUTPUT_DIR / f'{stem}_gpu{gpu_id}_{index:03d}.mp3'
        output.write_bytes(audio)

        '''
        Save the exact replay request and generated ABC when available.
        若有產生樂譜，儲存完整重播參數及 ABC。
        '''
        if replay is not None:
            output.with_suffix('.json').write_text(
                json.dumps(replay, ensure_ascii=False, indent=2), encoding='utf-8',
            )
            if replay.get('abc'):
                output.with_suffix('.abc').write_text(replay['abc'], encoding='utf-8')
        print(f'GPU {gpu_id}: saved {output.name}', flush=True)
        return output
    OUTPUTS = [generate_on_gpu(GPU_ID)]

assert OUTPUTS and all(path.is_file() and path.stat().st_size > 1024 for path in OUTPUTS), '尚未取得有效音訊檔案。'
print('✓ 第 6 節完成：歌曲已生成並確認輸出音訊檔案。')


7. 試聽、樂譜與下載


In [ ]:
'''
Preview audio and render optional scores without loading GPU models.
試聽歌曲並顯示選用樂譜，不重新載入 GPU 模型。
'''
import shutil
import subprocess
from IPython.display import Audio, FileLink, SVG, display
tools = [name for name, enabled in [('abcm2ps', SHOW_SCORE), ('abc2midi', EXPORT_MIDI)]
         if enabled and shutil.which(name) is None]

if tools and not globals().get('SCORE_TOOLS_ATTEMPTED', False):
    SCORE_TOOLS_ATTEMPTED = True
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'abcm2ps', 'abcmidi'], check=False)

for output in OUTPUTS:
    print(output.name)
    display(Audio(filename=str(output)))
    display(FileLink(str(output)))
    replay_path = output.with_suffix('.json')
    score_path = output.with_suffix('.abc')
    if replay_path.is_file():
        display(FileLink(str(replay_path)))
    if not score_path.is_file():
        print('No ABC score returned (for example, COT=off).')
        continue
    display(FileLink(str(score_path)))

    '''
    Use abcm2ps to export a standard SVG staff notation.
    使用 abcm2ps 將 ABC 排版成標準五線譜 SVG。
    '''
    if SHOW_SCORE and shutil.which('abcm2ps'):
        prefix = f'{score_path.stem}_sheet'
        rendered = subprocess.run(
            ['abcm2ps', '-g', '-S', '-O', prefix, score_path.name],
            cwd=OUTPUT_DIR, capture_output=True, text=True,
        )
        if rendered.returncode:
            print('Score renderer:', rendered.stderr[-400:])
        else:
            for score_svg in sorted(OUTPUT_DIR.glob(f'{prefix}*.svg')):
                display(SVG(filename=str(score_svg)))
                display(FileLink(str(score_svg)))
    if EXPORT_MIDI and shutil.which('abc2midi'):
        midi_path = score_path.with_suffix('.mid')
        converted = subprocess.run(
            ['abc2midi', score_path.name, '-o', midi_path.name],
            cwd=OUTPUT_DIR, capture_output=True, text=True,
        )
        if converted.returncode == 0 and midi_path.is_file():
            display(FileLink(str(midi_path)))
        else:
            print('MIDI conversion unavailable for this ABC:', converted.stderr[-300:])

print('✓ 第 7 節完成：試聽、樂譜及可用下載連結已處理。')


8. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'music'
REMOTE_FIELDS = ['task', 'style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot', 'abc']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\x9dXmS\xdc\xd6\x15\xfe\xce\xaf\xb8\xcd\x17i\xc7X\x80i\xdd\x84\xe9f\x06c\x9a2\x8d\r\x0383\x19\xc7\xb3#\xa4\xbb\xbb\nZI\xd5\xbd2l\x19f01\x01\x02\x18&c\xf0\x0b\xc4\xd8\x1e\x9bbcp\xec\xc4\r!\xd8\xfe1E\xda\xe5S\xffB\xcf\xb9W\xda\xd5.v\x9avm`u\xef=\xe7\x9e\x97\xe7\xbcIQ\x94\x96\xbe\x92\xe7\xfa\x9c\xf0"%\xb65\xe2\xeb\xbeE\x19q(5\xa9I\xf2\xae\x0f\x1b\x16#\x8c\x1a\xdcr\x1d\xad%\\|\x1e\xce<\x8e6\x9eE{\xaf*\xdb\x0b\xe1\xe1r4?u\xbc1U\xb9{=\x9c}\r\x8f\xe1\xc1\xce\xbf\xa6\xa6[\x14\xe0lI\xce_2\xd7I\xbes\xabD\x93\xef>\xfd[@\x19g-\xe2\xf0 \xd5M\xa2;&\x19\xf3-NI\xde\xb2A\x8c\xc01\xa9O\x0c\xd7\xc9[\x85\xc0\x07\x81\xc6\\\x7f\xd4r\nZKuo*|\xbeC\xfe\xaa\x17\n6%\xe1O\x8f\x8f^oT\xd6\xf7\x8e\x17\xaf\x1f\xed\xef\x820\xd1\xd3\x9b\xd1\x83\xb9D\x92\xbc\xef\x96\x88\xa7\xf3"hH\xe2\xeb\x07\xe0Qn\xd0\x92n\xd9\xc9\xb2\xe7\xda\x96QNmh\x9e\xee3\x90"\xde?W\xe6\x94\r\x88%)\xf8g\xbam\x99:\x88\x8c\x16,P\x87\xfa:\x9a\nL\xc69\x88\xca\xc8\x08\x053\xd2D]X"\x9f\x0c\\\x12\x9a\x8097\x7f\xaa>X\x14\x0b\xd1\x8d\xed\xea\xceZ8\xbf\x14\xce\xcc\x1d?\xb9U\xdd\x9d\xaf\xdc\xdc\x8c\xe6V\xaa\xdb\xbb\xe1\xde\xddD\x13\x9d\xc1\xc5\x1c\tr}\xe7\x89\xe5\x90\x81\xfe\xc1\xe1\xa1V\xa2\xc4+\xe1\xf2\xd3\xeaW\xaf%Me\xfa\x80\xb4\x93hn\x8dt\x00y\x8d\xb6\xa7\x7f\x18\tU%\x1f\xd8\xb6\x02\xa4%j\xbbf\x19\xbf\xb9\xf9\xbc\x92\x11^8\x7fi\xb0{\xb8\xaf\xff"\xf9\x18X\xe0\xc2\xd0p\xef\xc0\x10>%\xee\n\x18%:a\xa0\x0f8\xc0\r\xb8\x17p\xc0\x8b\x8d\x0e\x1b\xa1|\x8cR\'e\r\x06\xaa\xae\\\x8f\x9e=\x90:U\xee~\x1d\xbd|[\xb9\xb9]Y\xdf\x0fW\x16\xab\x87\xfb\xe1\xecA\xf5\x87\xd9h\xedN\xf8\xe8M\xa2k\xff\xa5\xe1\x81K\xc3\xb9\xf3}\x83$K\x06\xfbA\xec6\xa2\x94\x03z&\'\xafc\xe9#Zi\xd4\xb4|\x15\x9cE\x1d\xce\xb2\xc3~@[\t\x1d\xb7\x18\xcf\xb9\xa3\xe21#%\xef\xf7P \xdd\xb6\xcb\xc4v\x11s\x044\xf1OS\x13`g\x92\xees=\x84\x19\xe8/\xee\x92B`\x99\x94\x94\x02f\x19)]\x00|[k\x00\xf9\xea\xd65D\xe1\x0c@\x0fU\xa9N\xcd\x1c\xbd\xdd\x8bn\xfe\x0c\xe8\x13l\xa2\xed\xe9\xea\xee\x06\xe2r\xfa@\xaa\x1d=\xfc\xee\xe8\x97\x7f&\xea\x89kr\x9c\x8esP\x0f\xd1\xa8\x02Un\xa0{\xf8/\x19\xcd\x87h\x10[*u\x0c\xd7\x04#g\x95\x80\xe7O\x7f\xa8d4\xc6}\xcbS3\xc4\xca\x93\x84\x80P\x1b\xbcQ\x87\x87\xe3r\x92b\x0fQ\x8cN\xff]Vz\x18\x1c\x1d-\xaf\x1c\xbdY\xaf\xd3Gw\xa6\xff}\xb8\x88\x87\xa2\xd9y\x89\x1e\x82\xf0\x10\xe8\x91\xf8\x10\x10\x12\x82\x9f\x0b,\x1b\xed\x16#\x9a\x94tn\x14\x11\xd7\x18\x01p\x81eX\xbaM\xd0Q\x9a\xe1y\xa4{\xa0\x0f\xbc\xff\xcbAeg\xa1\xf2l+\\\x99\xabo\x85{\xb7\xa3\xb5\x9f\xa3\xfb\x87\x908\xc0l1\xe0w\x16\xa2\x17\xd3\x89\x95\xe2Kr\x10b:\xd8i\xa2\x85\xc0Ga\xbclS\xa5\x0bP\xf9\xf9\xa7\xbd\xa0\x8f]\xf6-\x83\xc1\xc2\xa7\x9f\x0f\xf6\xf5`<\xe8#\x06<\xd6\x8d\xd0*\t\r\x97\xc32(\nG\xcc@:\x14\x16\x12\xb8\xb7"k\xea1\xc1\x1a\x00\x1fS\xd9\xa5\x1c\x83\x8c\x88\xfc/\xe4\x86z{\xcf\xe39\xb9\x10?\xc1\x89\x114C\x8eY\x7fG\xc1:bJVvx\xb1y\x0b\xdc \x00\x9c\x83\xc4\x00\xc6\x835\xa5\xe4u\x8aP\xf4:s#\x16\x07\xb9\xc4\xc9\x8f\xce\xb4\xb6LJ\xa3\x7f"\xf1\x07\x06v\xe0?%&eV\xc1\xd1\x11\xb3\xe0\xf2\xab\x10uc\x16/\x02_\xc2\xb8\xee\x8b4\xa3C\x1a\x1f#%\xd7\xa46\xf1|\xd7\xa0\x0c\xc2P\x825Z\x9c\x05\x17G\x1bK\xe1\xc2\x9d\x18\x9b\xbb\x8b\xd1\xfaK\x00\xc1\xd1\xfe\xd2\xf1\xecR\xb4\xf6}\xb8\xba\x19.\xacF\xdb\x0f\xc2{\x0b\x90\xa4 \xdb\'>1i>\x89\x07\x9as\x9d\\\xc1\x0bT\xf8\xc9Yf\xa6\xabE\xea\r\xa7\xf0o\xb7\x81\xb7\xca\x1a\x02\x19Ja\x10s\x06\x80\x030A\xa8cz\xae\xe5pM\x9c\x0c\x97\xd7\xc27\xb7\x8e\xa7\xee\x84\x87\xafD>\x0c\xbf_\x02,b*\x87b\xf3d\x13I\xf0\xfe4w\xdd4}d\x9f%y\xa5\xc8\xb9\xd7\xd5\xd6\xd6q\xe6\x8fZ;\xfc\xeb\xe8\x9a\x10\x99\xf1\xb2\x94\xeb\xca\xa4\xa4`\xc1H\xc9\xe2h\xb4l\xad\xfeh\x9e\xcb\xb8\x9a\xf0:E\x946\xe14p\x07V\xael\x1a\x80\xad\xa2\x80\x81\x91\xb3g\xdb3\x8d\x0c5_\xb7\x18E\x8f\xe6\xc0\x01<`\xaa<\xf0\xa5;\x02\xf7\xc3u\xf5\x93\xc8V\xcd\\V,S\xb9"\xcex>\x98A\xcdc\x0e\'\x13R\xe0\xc9.\xa4$\x13\x92|R:\x15\x00\xd7J\xf2v\xc0\x8aIBK[c\xc0\x85x\xc5\x10DB\x94\x81\xd6 \xe1S\x8f\xea\x02\x13\xb6[\x80xfL/@y\xa5\xbaQ\xc4\xd2\xee:f\xec\x04\xc4\xc4f\xb4\xf9\xb8\xfa\xe4\xa1\x84\xc5\xf1\xd4\xcb\xf0`\x0b`q|\xedm8\xb3\x14=_\xae\xfc\xe3[\xc0G\xf5\xd1\xac\xcc\xdaq\x06\xdf\xfe&\xba\xf6\xbc\xd9?&d1\xdb\x02\xb0f\x85\xdd\xb4\x92\x0b\x89\xc9u,\x03\xb2\xd7)\xd2y\xb6\xbd]\x1c\xb3u0/\x88\x88\xe5\xf5\xe4Iqd\xac\x08\xfd\xc0I&\x7f\xaa]\x11\xe3.};~z\x8a\xd4\x18\x15F1\x02\x1f+\x03\x91\xce\x11%\xad\x08\xbf\x80k\x1e\x8a<4\x17`\x8eq\x0fj\xbf\xc5\xed\xb2Vc\x10=}\x08\xe6\x88\xfb\x8b\x85\xa9hfA\x84\xc8Vt\xfbF\xe5\xe1A\xb8~/\xbc\xff"|\xf4"Z\xbd\x15\xde\xd8\x0c\xd7\xef\'&h\x16D^\x0bZ2\x0f\xaa!M\xc3\xaf@\x1b\xd1\x07\xee\x037C!\xd3K,;\x81(\xe9\x8aQ4Y\xc7_g\x8c\xbfw\xf0~\x1f\x12\xebG\x11\x8dM4\x12\x93\xb5sP[\xe4\t!\x9d"\xbfCk\x90\x85\x12bB\xfaQ\xbajG\xf13\x02\xf5j\xf4\xbf\x12\xcb\xa6\x03\xac-p\xac\x18\xbacP\x1b\x1f2\x8d\xdc\x84\xfcd0pP\xd9^\xdfw\xfd\x13\xb11!\x99N*\r"\x9f\xc0\xc7\xe9\x06l}\x9c%\x9d\xed\x8dW\xbd;\xf2\xa0K\x83PJ*>D\x8d:\x91R\xe8\x03\xf9\xfd\x83\xccd\xa6)\x1c\xd3\x9c\x7f\x03\xa8\xf1#6\x98M\xa9\xa7\xfeA\xaeb)\xaf\x0b)m1,\xbd\xfen[4\xe4\t:n\xc8\xd6\xbd\x83@\xe4\xfb\x9a\xd2\x94$\xce\xbbc\x8eh{\x1aju)\xb0\xb9\x05\x88\xc3L\xc1\xe0\xbbH\x1c\x985l\xbd\x0c\xe9\x82\xeb\xa2\xfab\xcc\xe8\x81i\xb92<\x8e\xf6\x17\xaa\x87\xf3\xe1\xe2L\xb8\xb2\x83\x15\xe3\xdb\xddp\xf9\xabhu\xbf:7{\xbc\xf9\x03d\x04\x9c\tD\x81O\xf1\xaf\xbcZ\x89\xeem4\'\x8a\xf7\x85E\xcd\x0c\xbf->\x00TR~\xac\x9d\xa9`\xe9\xf8\xb0]\x96\xe3L\xc3m\xef\x0b\x94\x12\x90\x81$\xf5\xdbG\x94\x0b}\x17zO\x7fF}\x06\xed\x02\xf0\xd6\xda\xbf\xf0\xbfpz\\\x87CJ9=\\\xf6(\x94\xf0\xda\xf1S\xf5\x1b\x8a\x90\x9e\x80\xea\xb2\x92>\xab\\\xd1DWGS88\x05\x97 O\xfcQ\xd2\x1c\x0cI\x98\x12?N\xdf aj\x16Q\xe5\xdc\x92\x95\x7f4\xa8\xd0:\xd8!#\x87\x97\x11<\xa7\xa2Z\x92C\xdc%\xc6\x8c4\x8b\xe5j\x0eR3\x00\xe9&\x80\x05\x0edF\x18\x00\x01U1<d\xdf\xa2I\x95\x91\x0cS\x8a\r\xbd\xb6Z\xe3\xc9\xa9\x9f\x13;j\xa6\xb9N\x01\x13\xecX\x04\x04\x05\x9e\x04\xb2\xf0\t\x10\xe7\xbbf`X#\x90\x97e\x1b^C\x1f\xa3\xe8rN\x93\xec\x1c\xce}\x1d\xce=>z\xfb]e\xf5\x0e\xb90\xd0I$\xe8\x00}\xe1\xf2s\xc0ce\xf9\r6\x0f\xa2\t\x87\xd1\x02\xb6N\xb4\x0f\xe2\xee,tI\xe3)\xa8\xe1\x08\xc1\x11} ~\x19CE5):K\xc6\xb8\x18\x89\x05\x92!\x9b\t\xfd\xeaQ\x0f\x19\xa8F\x1b\xbb-\xc7\xc1\xdfj\x9c\xfa\xc4}m%\x8f\x160\xfd%O\x9dJ&\xd3Zcr\x11\x92k\x1a\xab\xb1\xab\xeaf\xb2\xa9\xa3\x8a\xa7\x0c\x0cc\xbfo\xff\xe8\xecI\x7f\xd1\x92\xc7\xcb\xc2$5\x14%a&\x02\xfa\x84\xc6\x98\xfc5T\x95\xfd\xba\xf2\xff\xaf\xf6X5t\x0f\x8b\xab\xe8\xb7\xdb\xf0>\xe5=J7\xb6\x8f6\xb6\x8b\x1c\x07\xcd\xc0\xb1 1\x88\xf7\x02\x8e\x0e\xf1Y{5!\xdbJ\xd9PKd\xc0\xbc\x95\xea#\x11\x04\xa2\xb7\x95#\x88lo\xa1}\x91/\t\xc2\x95\xa5fX\xc0\x00PJ&\xb2x\xbe\xbc\xd8}\xa1\x17\x07/\xd8\x80k\xc5\x0c*\xcfZ\x8eI\xc7\xe1pG\xaaKQSsk\x1b8g\x02\xe9&\xb1IN\\\x94\x9b\x10t]\xed\x9d\xe6\xa4&\x10\xa0\x89I\x15b\xa5\x9e\xf9%\xebS\t\xefx\xc2\xce\x92\xff\x9d{\x8a^\x13oWr2!H\x185Z|H\x8f\x03\x93\x8e\xeb\x06O\x10\x93Lz\x08\xc0\xa4\xf1\x97\xb3\xf2X\x11\xe6|\xfd*\x14u\x1dbV\xda\xbf\xba\xf08\xda\x98\xaf\xdc\x14-\xa4\x88?\xe8\x99\xc2\xeb/\xc3\xdd\xdb\xe1\xdeb\xb4\xfac\xbaZ\x84\xcb\xdf \xa3f\'\x00\x9a\xe2\xbb\xc1\xc58\xce"F\xea\xc6I\xb4\x812\x95c\x01\x94\xb1qU\xd1$\xacb\x15y\x03\xc4E\'\x8e(7\x83\x92\xc7T\xc9\xba\x15F\x10\x06\x9d_Ng\x86ee\xff\xacC\xe5m\x15vwx\xf6L\x06\xb7\x1bg\xef:b\x1b\xda\x0e\xc9M6;8v6\xb53\xef\x94U\x9cK\x8b*\x99\\\x16\x0c\xae\x9c\xbc:\xf3+\x83\x02\x03\x9f\x99d"\xbe\x07\xa3c\xf2\x1d\x8d\x89Oy\xe0;\xb14\xf1K\x9c@\xce\x92`\x98B2W"kt\xf3(\xb4$\xc4\xe2\xac\xfej\xc7\x17\xb8\xf0|z\xd5\x82\xb1\x12[7\x18\xeb\x97\x9f\x86\x1b\xdbG\xfbS\xb5h\x13S\x03,\x1co\xad\xd5GJ\x91\xa4e\xe9?z\xb3^}\xb2U\xbd\xf6\x9a\xf4\x00\x07"\xc7\xd1\xc67>C\x80\xf2\xcb\xcd\xf3\xa5|\xbf\x95\xb9\xf2\x1f\xbe\xf2\x9e\x14').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['music']
REMOTE_MAX_IMAGES = 0

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            gpu_id = globals().get('GPU_ID', 0)
            selected = globals().get('YUE_DEVICE_NAME')
            port = globals().get('PORTS', {}).get(gpu_id)
            backbone = globals().get('BACKBONE')
            if not selected or port is None or backbone is None:
                return False, 'YuE2 配置或模型路徑尚未初始化。', []
            url = f'http://127.0.0.1:{port}'
            with urllib.request.urlopen(url + '/health', timeout=3) as response:
                health = json.load(response)
            with urllib.request.urlopen(url + '/props', timeout=3) as response:
                props = json.load(response)
            if health.get('status') != 'ok' or Path(props.get('model', '')).name != Path(backbone).name:
                return False, 'YuE2 服務或模型與指定配置不一致。', []

            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            arguments = [str(arg) for arg in process.args] if process is not None and process.poll() is None else []
            if not arguments:
                # A server started in a previous notebook kernel can survive as a Linux process.
                # 前次 Notebook Kernel 啟動的服務可能仍在 Linux 背景執行。
                for entry in Path('/proc').iterdir():
                    if not entry.name.isdigit():
                        continue
                    try:
                        candidate = [part.decode(errors='replace') for part in (entry / 'cmdline').read_bytes().split(b'\0') if part]
                    except (OSError, PermissionError):
                        continue
                    if candidate and Path(candidate[0]).name == 'yue-server' and '--port' in candidate:
                        port_index = candidate.index('--port') + 1
                        if port_index < len(candidate) and candidate[port_index] == str(port):
                            arguments = candidate
                            break
            if '--device' not in arguments or arguments.index('--device') + 1 >= len(arguments):
                return False, 'YuE2 服務可回應，但無法驗證實際 Vulkan 裝置；請重新啟動指定設備的服務。', []
            actual_device = arguments[arguments.index('--device') + 1]
            if actual_device != selected:
                return False, f'YuE2 使用 {actual_device}，與設定 {selected} 不一致。', []
            return True, f'YuE2 Vulkan 後端已就緒（{selected}）。', [selected]

        import torch
        if not torch.cuda.is_available():
            return False, 'PyTorch ROCm／CUDA GPU 不可用，請先執行模型初始化 Cell。', []
        selected_gpus = sorted(set(globals().get('MODEL_GPU_IDS', [0]) + [globals().get('AUX_GPU_ID', 0)]))
        available = torch.cuda.device_count()
        if any(type(i) is not int or i < 0 or i >= available for i in selected_gpus):
            return False, f'指定的 GPU {selected_gpus} 超出 PyTorch 可見範圍（{available} 張）。', []
        devices = [f'cuda:{i} {torch.cuda.get_device_name(i)}' for i in selected_gpus]
        if REMOTE_KIND == 'image_multi':
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／記憶體管理器尚未初始化。', devices
            if state.get('device') != f"cuda:{globals().get('MODEL_GPU_ID', 0)}":
                return False, 'Qwen Transformer 的 GPU 配置不一致，請重新載入。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa Pipeline 尚未完整載入。', devices
            for name, module, gpu_id in (
                ('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)),
                ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 0)),
            ):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{gpu_id}'):
                    return False, f'{name} 並未載入指定的 cuda:{gpu_id}。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any('cuda' in str(item.get('type', '')).lower() or
                       'cuda' in str(item.get('name', '')).lower() or
                       'rocm' in str(item.get('name', '')).lower()
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 未回報 GPU 後端，已拒絕 CPU-only 推論。', devices
        return True, 'GPU 後端已就緒（不表示所有權重常駐 GPU）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    if REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        cot_mode = str(fields.get('cot', '')).strip().lower()
        if cot_mode not in ('off', 'melody', 'full'):
            raise ValueError('YuE2 COT 只支援 off、melody、full')
        if cot_mode == 'off' and str(fields.get('abc', '')).strip():
            raise ValueError('COT=off 時不能提供 ABC 樂譜')
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'notebook-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Host disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    namespace['ABC_PATH'] = ''
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                if str(inputs.get('cot', '')).lower() != 'off':
                    scores = [path.with_suffix('.abc') for path in results
                              if path.with_suffix('.abc').is_file()]
                    results += [sidecar for path in list(results)
                                for ext in ('.json', '.abc')
                                if (sidecar := path.with_suffix(ext)).is_file()]

                    '''
                    Convert ABC to MIDI and SVG only when the optional tools exist.
                    有安裝樂譜工具時，自動將 ABC 轉成 MIDI 與 SVG；工具缺少不影響音樂生成。
                    '''
                    for score in scores:
                        midi = score.with_suffix('.mid')
                        if shutil.which('abc2midi'):
                            try:
                                converted = subprocess.run(
                                    ['abc2midi', score.name, '-o', midi.name],
                                    cwd=score.parent, capture_output=True, text=True, timeout=90,
                                )
                                if converted.returncode == 0 and midi.is_file():
                                    results.append(midi)
                            except (OSError, subprocess.TimeoutExpired) as error:
                                print('Optional MIDI conversion skipped:', type(error).__name__)
                        if shutil.which('abcm2ps'):
                            try:
                                prefix = score.stem + '_sheet'
                                rendered = subprocess.run(
                                    ['abcm2ps', '-g', '-S', '-O', prefix, score.name],
                                    cwd=score.parent, capture_output=True, text=True, timeout=90,
                                )
                                if rendered.returncode == 0:
                                    results.extend(sorted(score.parent.glob(prefix + '*.svg')))
                            except (OSError, subprocess.TimeoutExpired) as error:
                                print('Optional SVG conversion skipped:', type(error).__name__)
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                for audio_file in paths:
                    if audio_file is None:
                        continue
                    audio_path = Path(audio_file)
                    if not audio_path.name.startswith(f'remote_{job_id[:12]}'):
                        continue
                    for suffix in ('.json', '.abc', '.mid'):
                        audio_path.with_suffix(suffix).unlink(missing_ok=True)
                    for sheet in audio_path.parent.glob(audio_path.stem + '_sheet*.svg'):
                        sheet.unlink(missing_ok=True)

            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 8 節完成：選用 FIFO／ngrok API 已啟動。')


9. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_YuE2_音樂生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 9 節完成：API Schema 和 client 說明已顯示。')
